# 1. Imports

In [ ]:
import asyncio
import os
import sys
import warnings

import pandas as pd
from openai import OpenAI

sys.path.insert(0, "")  # Change your AutoDDG src path here

from autoddg import AutoDDG, GPTEvaluator
from autoddg.evaluation import QuestionDatasetEvaluator
from autoddg.url import dataset_information, dataset_download, dataset_dictionary
from autoddg.utils import get_sample

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 2.URL Define

In [ ]:
url = '' # Change your dataset URL here

# 3.Download related data from URL

## Windows user for general info

In [ ]:
# import asyncio
# import concurrent.futures
# import truststore
# truststore.inject_into_ssl()

# if sys.platform == 'win32':
#     asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
# def run_in_proactor(url):
#     loop = asyncio.ProactorEventLoop()
#     try:
#         return loop.run_until_complete(dataset_information.fetch_full_dataset_info(url))
#     finally:
#         loop.close()

# with concurrent.futures.ThreadPoolExecutor() as pool:
#     general_info = pool.submit(run_in_proactor, url).result()


## Mac user for general info

In [ ]:
!playwright install # Install Playwright browsers (Chromium, Firefox, WebKit) for first-time use

In [ ]:
general_info = await dataset_information.fetch_full_dataset_info(url)

In [ ]:
df = dataset_download.download_dataset(url)
files = dataset_dictionary.get_attachments(url)
info = dataset_dictionary.get_dictionary(files, general_info, df.shape)

In [ ]:
# Load dataset
title = general_info.loc[0, "name"]
original_description = general_info.loc[0, "description"]
tags = general_info['domain_tags'].values[0] if 'domain_tags' in general_info.columns else None

In [ ]:
df.head()

In [ ]:
# Sample rows for processing
sample_df, dataset_sample = get_sample(df, sample_size=100)

# 4. Initialize AutoDDG

## Option A: Using OpenAI API (Recommended for quick start)

In [ ]:
# Option A: OpenAI API
my_api_key = "YOUR_OPENAI_API_KEY"  # Replace with your key
client = OpenAI(api_key=my_api_key)
model_name = "gpt-4o-mini"

# Initialize AutoDDG with API client
auto_ddg = AutoDDG(client=client, model_name=model_name)


## Option B: Using Local LLM (Qwen, Llama, etc.)


For local LLM support, install the optional dependencies first:

pip install git+https://github.com/VIDA-NYU/AutoDDG@main transformers torch

In [ ]:
# Option B: Local LLM
from autoddg.llm import LocalLLMClient

# local_client is used for QuestionDatasetEvaluator in Section 8
local_client = LocalLLMClient(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",  # or any HuggingFace model
    device="cpu",  # or "cuda" if GPU
    torch_dtype="float32",  # or "float16", "bfloat16"
)

# AutoDDG setup
auto_ddg = AutoDDG(
    client=None,
    model_name="Qwen/Qwen2.5-1.5B-Instruct",  # or any HuggingFace model
    use_local_llm=True,
    local_llm_device="cpu",  # or "cuda" if GPU
    local_llm_dtype="float32",  # or "float16", "bfloat16"
)

# 5. Prepare Context


In [ ]:
# Generate basic structural profile
basic_profile, structural_profile = auto_ddg.profile_dataframe(df)

# Generate topic
data_topic = auto_ddg.generate_topic(
    title=title,
    original_description=original_description,
    dataset_sample=dataset_sample,
    tags= tags
)

# 6. Semantic Analysis and Constraints


In [ ]:
# Sequential mode (default) - works with both OpenAI API and Local LLM
semantic_profile_details = auto_ddg.analyze_semantics(sample_df, data_dict = info)

# Combine with structural profile
semantic_profile = "\n".join(
    section for section in [structural_profile, semantic_profile_details] if section
)

print("Semantic analysis completed (sequential mode)")

In [ ]:
constraints_prompt, constraints = auto_ddg.extract_constraints(
    info,
    dataset_sample=dataset_sample)

# 7. Generate Descriptions

In [ ]:
# General description
prompt, description = auto_ddg.describe_dataset(
    dataset_sample=dataset_sample,
    dataset_profile=basic_profile,
    use_profile=True,
    semantic_profile=semantic_profile,
    use_semantic_profile=True,
    data_topic=data_topic,
    use_topic=True,
    tags = tags,
    data_dict=info
)

# Search-focused description
search_prompt, search_focused_description = auto_ddg.expand_description_for_search(
    description=description,
    topic=data_topic,
)

In [ ]:
print(description)

In [ ]:
print(search_focused_description)

# 8.Asking question about dataset

## Option A: Using OpenAI API

In [ ]:
question = "" #Question here

evaluator = QuestionDatasetEvaluator(client,model_name = model_name)
result = evaluator.evaluate_relevance(question, description)
print(result)

## Option B: Using Local LLM

In [ ]:
question = "" #Question here

evaluator = QuestionDatasetEvaluator(local_client)  
result = evaluator.evaluate_relevance(question, description)
print(result)

# 9. Evaluate Quality (Optional)

In [ ]:
try:
    auto_ddg.set_evaluator(GPTEvaluator(gpt4_api_key=my_api_key))

    # Score descriptions
    general_score = auto_ddg.evaluate_description(description)
    search_score = auto_ddg.evaluate_description(search_focused_description)

    print("Score of the general description:", general_score)
    print("Score of the search-focused description:", search_score)
except Exception as e:
    print(f"Evaluation skipped: {e}")
    print("Note: Evaluation requires OpenAI API access")

# 10. Evaluate Pairwise (Optional)

In [ ]:
from autoddg.evaluation.pairwise import PairwiseEvaluator

original_search_focused_description = """""" # Add original search-focused description here
original_description = """""" # Add original description here
pairwise = PairwiseEvaluator(client=client, model_name=model_name)
print(pairwise.compare(original_description,description))
print(pairwise.compare(original_search_focused_description, search_focused_description))